In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
from matplotlib.colors import LogNorm
from collections import Counter
import random
import os

In [ ]:
def plot_heatmap(A, N, log=False):
    heatmap = np.zeros((N, N))
    size = 20
    for x, y in A:
        heatmap[x][y] += 1
    
    # Normalize by the maximum value
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    # Plot the normalized heatmap with custom aspect ratio (3 height, 4 width)
    plt.imshow(heatmap, cmap='gray_r', interpolation='nearest', origin='lower', aspect=3/4)
    
    # Add x and y axis labels
    plt.xlabel('Destination rack', fontsize=size)
    plt.ylabel('Source rack', fontsize=size)
    
    # Increase tick label font size
    plt.tick_params(axis='both', which='major', labelsize=size)

    # Add colorbar, adjusting its size relative to the heatmap
    cbar = plt.colorbar(fraction=0.046, pad=0.04)  # Adjust fraction and pad to control the size
    cbar.ax.tick_params(labelsize=size)
    
    plt.show()

    if log:
        # Plot the log-scaled heatmap with custom aspect ratio
        plt.imshow(heatmap, cmap='gray_r', interpolation='nearest', norm=LogNorm(), origin='lower', aspect=3/4)
        
        # Add x and y axis labels
        plt.xlabel('Destination rack', fontsize=size)
        plt.ylabel('Source rack', fontsize=size)

        # Increase tick label font size
        plt.tick_params(axis='both', which='major', labelsize=size)

        # Add colorbar, adjusting its size
        cbar = plt.colorbar(fraction=0.046, pad=0.04)  # Same adjustment for the colorbar
        cbar.ax.tick_params(labelsize=size)
        
        plt.show()
        
def count_distinct_nodes(A):
    distinct_nodes = set()
    for x, y in A:
        distinct_nodes.add(x)
        distinct_nodes.add(y)
    return len(distinct_nodes)

def build_graph(commodities):
    graph = defaultdict(set)
    for u, v in commodities:
        graph[u].add(v)
        graph[v].add(u)
    return graph

def dfs(node, graph, visited, component):
    stack = [node]
    while stack:
        n = stack.pop()
        if n not in visited:
            visited.add(n)
            component.append(n)
            stack.extend(graph[n] - visited)

def find_connected_components(commodities):
    graph = build_graph(commodities)
    visited = set()
    components = []

    for node in graph:
        if node not in visited:
            component = []
            dfs(node, graph, visited, component)
            components.append(sorted(component))

    return components

In [ ]:
def generate_traffic_microsoft(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, seedValue_, gamma_, commo_list):
    print(f'Running Python function with arguments: {Nrack_}, {Hosts_p_rack_}, {load_}, {time_}, {Gbps_rate_}, {Nactive_}, {workload_}, {network_}, {gamma_}, {seedValue_}')
    
    # Convert inputs to appropriate types
    Nrack_ = int(Nrack_)
    Hosts_p_rack_ = int(Hosts_p_rack_)
    load_ = float(load_)
    time_ = float(time_)
    Gbps_rate_ = int(Gbps_rate_)
    Nactive_ = float(Nactive_)
    seedValue_ = int(seedValue_)
    
    np.random.seed(seedValue_)
    random.seed(seedValue_) 

    Nrack = Nrack_
    Hosts_p_rack = Hosts_p_rack_
    H = Nrack * Hosts_p_rack  # number of hosts
    loadfrac0 = load_  # fraction of theoretically possible load
    
    totaltime = time_  # seconds
    rate = Gbps_rate_
    linkrate = rate * 10e8 / 8  # bytes per second
    Nactive = Nactive_
    gamma = gamma_
    
    filename = f'{network_}/{workload_}_{gamma:.2f}gamma_{100 * loadfrac0:.2f}percLoad_{int(totaltime)}sec_{Nrack}N_{Hosts_p_rack}hpr_{H}hosts_{rate}Gbps_{Nactive:.2f}Nactive_seed={seedValue_}.htsim'

    if os.path.exists(filename):
        print(f"File {filename} already exists. Skipping the process.")
        return

    print(f'Running Python function with arguments: {Nrack_}, {Hosts_p_rack_}, {load_}, {time_}, {Gbps_rate_}, {Nactive_}, {workload_}, {network_}, {seedValue_}')
    
    
    H_active = int(np.ceil(H * Nactive))
    print(f'H_active = {H_active}')
    
    Ncons = H_active * (H_active - Hosts_p_rack)  # number of possible connections
    srcdst = np.zeros((Ncons, 2), dtype=int)
    cnt = 0
    cdf = 0
    tmcdf = np.zeros(Ncons)


    # Initialize the probability list
    probabilities = []

    pair_counts = Counter(commo_list)
    total_pairs = len(commo_list)


    for a in range(H_active):  # sources
        for b in range(H_active):  # destinations
            if a // Hosts_p_rack != b // Hosts_p_rack:
                # Store the source-destination pair
                srcdst[cnt] = [a, b]

                # Step 2: Determine the frequency of the pair (a, b)
                count = pair_counts.get((a // Hosts_p_rack, b // Hosts_p_rack), 0)  # Get count or 0 if not in commo_list
                p = count / total_pairs if total_pairs > 0 else 0  # Calculate probability based on frequency

                probabilities.append(p)
                cnt += 1


    # Convert the list to a numpy array for further processing
    probabilities = np.array(probabilities)

    # Normalize the probabilities to ensure they sum to 1
    probabilities /= probabilities.sum()

    # Calculate the cumulative distribution function (CDF) using np.cumsum
    tmcdf = np.cumsum(probabilities)

    print(f"len(srcdst) = {len(srcdst)}")
    print(f"tmcdf = {tmcdf}") 
    print(f"len tmcdf = {len(tmcdf)}")

    # Load flow size distribution from CSV
    if workload_ == 'DM':
        flowdis_data = np.loadtxt('../_flow_dis/DM.csv', delimiter=',')
    elif workload_ == 'HD':
        flowdis_data = np.loadtxt('../_flow_dis/HD.csv', delimiter=',')
    elif workload_ == 'WS':
        flowdis_data = np.loadtxt('../_flow_dis/WS.csv', delimiter=',')
    else:
        raise ValueError('Unknown workload specified')

    flowsize = flowdis_data[:, 0]
    flowcdf = flowdis_data[:, 1]

    print(f"data = {flowdis_data} {type(flowdis_data)}")
    print(f"flowsize = {flowsize}")
    print(f"flowcdf = {flowcdf}")

    
    avg_flowsize = np.sum(flowsize[1:] * np.diff(flowcdf))  # bytes/flow

    lambda_host_max = linkrate / avg_flowsize  # flows/second per host
    lambda_host = loadfrac0 * lambda_host_max  # flows/second for each host

    lambda_network = H_active * lambda_host  # flows/second for the entire network

    nflows_est = int(np.ceil(lambda_network * totaltime))
    flowmat1 = np.zeros((nflows_est, 4), dtype=np.int64)

    print('Getting PRIO flow start times...')
    crt_time = 0
    cnt = 0
    while crt_time < totaltime:
        next_time = -np.log(1 - np.random.rand()) / lambda_network
        crt_time += next_time
        if cnt >= flowmat1.shape[0]:
            flowmat1 = np.vstack([flowmat1, np.zeros((flowmat1.shape[0], 4), dtype=np.int64)])
        flowmat1[cnt, 3] = int(crt_time * 1e9)  # nanoseconds
        cnt += 1

    flowmat1 = flowmat1[:cnt]

    # ind = np.where(flowmat1[:, 3] > totaltime * 1e9)[0]
    # if len(ind) > 0:
    #     flowmat1[ind[0]:, :] = 69  # zero out flows beyond the total time

    print('Getting PRIO flow sizes...')
    randvect = np.random.rand(flowmat1.shape[0])
    indices = np.searchsorted(flowcdf, randvect)
    flowmat1[:, 2] = flowsize[indices]

    print('Getting PRIO flow sources & destinations...')
    randvect = np.random.rand(flowmat1.shape[0])
    indices = np.searchsorted(tmcdf, randvect)
    print(f"indices = {indices}")
    flowmat1[:, 0:2] = srcdst[indices]

    actual_load_frac = np.sum(flowmat1[:, 2]) / (totaltime * H * linkrate)
    print(f'\n\nSpecified fraction of capacity = {loadfrac0:.3f}')
    print(f'Actual fraction of capacity = {actual_load_frac:.3f}\n')

    # Write the flowmat1 to the file in the appropriate format
    write_to_htsim_file(flowmat1, filename)

def write_to_htsim_file(flowmat, filename):
    with open(filename, 'w') as f:
        for idx, row in enumerate(flowmat):
            if idx == len(flowmat)-1:
                f.write(f"{row[0]} {row[1]} {row[2]} {row[3]}")
            else:
                f.write(f"{row[0]} {row[1]} {row[2]} {row[3]}\n")
    print(f"Data written to {filename}")


In [ ]:
# def get_commo_list(num_nodes, seed_val, theta=0.075, phi=0.40, theta_2=0.34, phi_2=0.30, density=12):
def get_commo_list(num_nodes, seed_val, theta=0.075, phi=0.40, theta_2=0.34, phi_2=0.30, gamma=0.5, density=12):
    # num_nodes = 100

    # theta = 0.30
    # phi = 0.05# tor


    # theta_2 = 0.30
    # phi_2 = 0.38# tor
    np.random.seed(seed_val)
    random.seed(seed_val) 
    
    num_commo = density*num_nodes*num_nodes
    num_super_hot_pairs = int(num_nodes * num_nodes * 0.3 / 100)

    all_pairs = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i != j]
    num_nodes_hot = int(num_nodes * theta)
    num_nodes_medium = int(num_nodes * theta_2)

    hot_nodes = random.sample(range(num_nodes), num_nodes_hot)
    medium_nodes = random.sample([n for n in range(num_nodes) if n not in hot_nodes], num_nodes_medium)
    
    hot_nodes = [18, 31, 37, 38, 44, 69, 75, 88]
    medium_nodes = [2, 3, 5, 7, 15, 19, 20, 22, 24, 27, 29, 32, 34, 35, 40, 41, 42, 43, 47, 51, 58, 61, 62, 68, 72, 73, 74, 78, 82, 87, 89, 98, 100, 101]
    print(f"hot nodes = {hot_nodes} {len(hot_nodes)}")
    print(f"medium_nodes' = {medium_nodes} {len(medium_nodes)}")

    super_hot_pairs_candidate = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i > j and (i in hot_nodes+medium_nodes and j in hot_nodes+medium_nodes)]
    super_hot_pairs = random.sample(super_hot_pairs_candidate, num_super_hot_pairs//2)
    for pair in super_hot_pairs.copy():
        super_hot_pairs.append((pair[1], pair[0]))

    commo_list = []
    prob1 = phi/num_nodes_hot
    prob2 = phi_2/num_nodes_medium
    prob3 = (1 - phi - phi_2) / (num_nodes - num_nodes_hot - num_nodes_medium)

    print(f"{prob1} {prob2} {prob3}")
    while len(commo_list) < num_commo:
        value = random.random()
        
        # if value <= 0.8:
        if value <= gamma:
            (src, dst) = random.choice(super_hot_pairs)
        else:
            # (src, dst) = random.choice(possible_pairs)
            while True:
                src = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                dst = random.choices(range(num_nodes), weights=[
                    prob1 if i in hot_nodes else 
                    prob2 if i in medium_nodes else 
                    prob3 for i in range(num_nodes)
                ])[0]
                
                if (src in hot_nodes + medium_nodes or dst in hot_nodes + medium_nodes) and (src,dst) not in super_hot_pairs:
                # if (src in hot_nodes + medium_nodes or dst in hot_nodes + medium_nodes):
                    break
            

        commo_list.append((src, dst))

    # Example: Printing the first few communication pairs
    print(commo_list)
    print(count_distinct_nodes(commo_list))
    print(f"num pair = {len(set(commo_list))} = {len(set(commo_list))/(num_nodes*num_nodes-num_nodes)*100}%")
    plot_heatmap(commo_list, num_nodes, True)

    cc = find_connected_components(commo_list)
    print(cc)
    a = [len(c) for c in cc]
    print(a)
    print(f"super_hot_pairs = {super_hot_pairs}")

    return commo_list

In [ ]:
Nrack_ = 108
Hosts_p_rack_ = 6
load_ = 0.08
time_ = 10.001
Gbps_rate_ = 40
Nactive_ = 1
workload_ = "HD"
network_ = "opera"
seedValue_ = 1
gamma_ = 0.50



In [ ]:
gamma_set = [0.00, 0.25, 0.50, 0.75, 1.00]
for gamma_ in gamma_set:
    commo_list_ = get_commo_list(num_nodes=Nrack_, gamma=gamma_, seed_val=seedValue_)
    generate_traffic_microsoft(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, seedValue_, gamma_, commo_list_)
